# LeetCode 2316 — Count Unreachable Pairs of Nodes in an Undirected Graph bằng Python

Link đề: https://leetcode.com/problems/count-unreachable-pairs-of-nodes-in-an-undirected-graph/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Graph

Skeleton LeetCode:

```python
class Solution:
    def countPairs(self, n: int, edges: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra "hai đỉnh không tới được nhau" ⇔ "hai đỉnh thuộc **hai thành phần liên thông khác nhau**" ⇒ chỉ cần **kích thước** các thành phần.
2. Đếm cặp khác thành phần bằng công thức `C(n, 2) − Σ C(sᵢ, 2)` hoặc bằng **tổng chạy** `ans += s · prefix`.
3. Tìm thành phần liên thông bằng DFS **lặp** / BFS (tránh tràn đệ quy với `n = 10⁵`) và bằng **Union-Find** (DSU) có nén đường + hợp theo kích thước.
4. Tránh các lỗi đếm: đếm cặp có thứ tự (gấp đôi), quên đỉnh cô lập, dùng DFS đệ quy trên đồ thị đường thẳng.
5. Mở rộng: duy trì đáp án khi **thêm cạnh trực tuyến**, và khi **xóa cạnh** (xử lý ngược bằng DSU).

## 2. Đề bài & ví dụ

Cho số nguyên `n` và đồ thị vô hướng `n` đỉnh `0 … n−1` với danh sách cạnh `edges`. Trả về số **cặp đỉnh khác nhau** không tới được nhau.

**Ví dụ 1:** `n = 3`, `edges = [[0,1],[0,2],[1,2]]` → `0`.

**Ví dụ 2:** `n = 7`, `edges = [[0,2],[0,5],[2,4],[1,6],[5,4]]` → `14`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `0 ≤ len(edges) ≤ 2·10⁵`; không có khuyên, không có cạnh lặp.

In [1]:
from typing import List
from collections import deque, Counter
import random, time

examples = [(3, [[0, 1], [0, 2], [1, 2]], 0), (7, [[0, 2], [0, 5], [2, 4], [1, 6], [5, 4]], 14)]
for n, e, exp in examples:
    print(f"n={n} edges={e} -> kỳ vọng {exp}")

n=3 edges=[[0, 1], [0, 2], [1, 2]] -> kỳ vọng 0
n=7 edges=[[0, 2], [0, 5], [2, 4], [1, 6], [5, 4]] -> kỳ vọng 14


## 3. Kiến thức nền

### 3.1 Thành phần liên thông

Quan hệ "tới được nhau" trong đồ thị vô hướng là **quan hệ tương đương** (phản xạ, đối xứng, bắc cầu) ⇒ chia tập đỉnh thành các lớp rời nhau — các thành phần liên thông. Hai đỉnh tới được nhau ⇔ cùng lớp.

### 3.2 Đếm cặp không có thứ tự

Số cặp không thứ tự trong `n` phần tử: `C(n, 2) = n(n − 1)/2`. Số cặp **cùng** thành phần: `Σ C(sᵢ, 2)`. Số cặp **khác** thành phần = hiệu, hoặc `Σ_{i<j} sᵢ·sⱼ`.

In [2]:
sizes = [4, 2, 1]                                # ví dụ 2: {0,2,4,5}, {1,6}, {3}
n = sum(sizes)
print("C(n,2) - sum C(s,2) =", n * (n - 1) // 2 - sum(s * (s - 1) // 2 for s in sizes))
print("sum_{i<j} s_i*s_j   =", sum(sizes[i] * sizes[j] for i in range(3) for j in range(i + 1, 3)))
print("(n^2 - sum s^2)/2   =", (n * n - sum(s * s for s in sizes)) // 2)

C(n,2) - sum C(s,2) = 14
sum_{i<j} s_i*s_j   = 14
(n^2 - sum s^2)/2   = 14


### 3.3 Danh sách kề và BFS/DFS lặp

Python giới hạn đệ quy ~1000 tầng; đồ thị đường thẳng `0 − 1 − … − (n−1)` làm DFS đệ quy sâu `n` tầng ⇒ `RecursionError`. Dùng ngăn xếp/hàng đợi tường minh.

### 3.4 Union-Find (DSU)

`find(x)` trả về gốc đại diện; `union(a, b)` gộp hai cây. **Nén đường** + **hợp theo kích thước** ⇒ gần như O(1) mỗi thao tác (O(α(n))). Mảng `size[root]` cho ngay kích thước thành phần.

In [3]:
parent = list(range(5)); size = [1] * 5
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb: return
    if size[ra] < size[rb]: ra, rb = rb, ra
    parent[rb] = ra; size[ra] += size[rb]
union(0, 1); union(3, 4); union(1, 4)
print([find(x) for x in range(5)], "kích thước thành phần của 0:", size[find(0)])

[0, 0, 2, 0, 0] kích thước thành phần của 0: 4


## 4. Ý tưởng nền tảng / trực giác

Không cần biết **cặp nào** không tới được nhau — chỉ cần **bao nhiêu**. Mỗi thành phần kích thước `s` "không tới được" mọi đỉnh ngoài nó: `s · (n − s)` cặp có thứ tự; chia 2 cho cặp không thứ tự.

Cách đếm gọn khi duyệt các thành phần lần lượt: giữ `prefix` = tổng kích thước các thành phần **đã gặp**. Khi gặp thành phần mới kích thước `t`: mọi đỉnh trong nó tạo cặp không tới được với mọi đỉnh đã gặp ⇒ `ans += prefix · t`, rồi `prefix += t`. Mỗi cặp khác thành phần được đếm đúng **một** lần (khi thành phần đến sau được xử lý).

Tính tay ví dụ 2: các thành phần theo thứ tự đỉnh nhỏ nhất: `{0, 2, 4, 5}` (4), `{1, 6}` (2), `{3}` (1).

| thành phần | t | prefix trước | ans += prefix·t | ans | prefix sau |
|---|---|---|---|---|---|
| {0,2,4,5} | 4 | 0 | 0 | 0 | 4 |
| {1,6} | 2 | 4 | 8 | 8 | 6 |
| {3} | 1 | 6 | 6 | **14** | 7 |

In [4]:
def components(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    seen = [False] * n; comps = []
    for s in range(n):
        if seen[s]: continue
        seen[s] = True; stack = [s]; comp = []
        while stack:
            u = stack.pop(); comp.append(u)
            for v in g[u]:
                if not seen[v]: seen[v] = True; stack.append(v)
        comps.append(sorted(comp))
    return comps

n, e, _ = examples[1]
ans = prefix = 0
for comp in components(n, e):
    t = len(comp); ans += prefix * t; prefix += t
    print(f"thành phần {comp}: t={t}, ans={ans}, prefix={prefix}")

thành phần [0, 2, 4, 5]: t=4, ans=0, prefix=4
thành phần [1, 6]: t=2, ans=8, prefix=6
thành phần [3]: t=1, ans=14, prefix=7


## 5. Cách 1 — Brute Force / baseline

### 5.1 Với mỗi cặp, BFS kiểm tra tới được — O(n² · (n + m))

In [5]:
def brute(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    def reach(src):
        seen = {src}; q = deque([src])
        while q:
            u = q.popleft()
            for v in g[u]:
                if v not in seen: seen.add(v); q.append(v)
        return seen
    return sum(1 for i in range(n) for j in range(i + 1, n) if j not in reach(i))

for n, e, exp in examples:
    print(brute(n, e), "OK" if brute(n, e) == exp else "SAI")

0 OK
14 OK


### 5.2 Baseline 2 — BFS một lần từ mỗi đỉnh — O(n · (n + m))

Từ mỗi đỉnh `i`, BFS cho tập tới được `R(i)`; số đỉnh không tới được là `n − |R(i)|`. Tổng chia 2. Vẫn O(n·(n + m)) ≈ `10⁵ · 3·10⁵` — quá chậm.

In [6]:
def bfs_each(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    total = 0
    for i in range(n):
        seen = [False] * n; seen[i] = True; q = deque([i]); cnt = 1
        while q:
            u = q.popleft()
            for v in g[u]:
                if not seen[v]: seen[v] = True; q.append(v); cnt += 1
        total += n - cnt
    return total // 2

for n, e, exp in examples:
    print(bfs_each(n, e), "OK" if bfs_each(n, e) == exp else "SAI")

0 OK
14 OK


**Nút thắt:** mọi đỉnh trong cùng một thành phần cho **cùng** tập tới được — BFS lặp lại cùng một công việc `s` lần cho thành phần kích thước `s`. Chỉ cần BFS **một lần mỗi thành phần** ⇒ O(n + m).

## 6. Ý tưởng thuật toán / bước đột phá

1. Xây danh sách kề.
2. Duyệt mọi đỉnh; mỗi đỉnh chưa thăm khởi động một DFS/BFS lặp, đếm kích thước `t` thành phần.
3. `ans += prefix · t`, `prefix += t`.

Hoặc thay bước 1–2 bằng DSU: gộp mọi cạnh, rồi đếm `size` của các gốc.

**Độ phức tạp:** O(n + m) thời gian (DFS/BFS) hoặc O((n + m) α(n)) (DSU); bộ nhớ O(n + m) (danh sách kề) hoặc O(n) (DSU — không cần lưu đồ thị).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy + tổng chạy (cách doocs 1)

In [7]:
def dfs_recursive(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    vis = [False] * n
    def dfs(i):
        if vis[i]: return 0
        vis[i] = True
        return 1 + sum(dfs(j) for j in g[i])
    ans = s = 0
    for i in range(n):
        t = dfs(i); ans += s * t; s += t
    return ans

for n, e, exp in examples:
    print(dfs_recursive(n, e), "OK" if dfs_recursive(n, e) == exp else "SAI")

0 OK
14 OK


Ngắn gọn, nhưng **đệ quy**: với đồ thị đường thẳng dài, độ sâu = độ dài đường đi ⇒ có thể `RecursionError` (mục 13, Lỗi 3). Với `t = 0` (đỉnh đã thăm), `ans` và `s` không đổi — mẹo gọn để không cần kiểm tra `vis` ở vòng ngoài.

### 7.2 Cách 3 — DFS lặp bằng ngăn xếp (cách doocs 2)

In [8]:
def dfs_iterative(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    vis = [False] * n; ans = prefix = 0
    for i in range(n):
        if vis[i]: continue
        vis[i] = True; stk = [i]; t = 0
        while stk:
            u = stk.pop(); t += 1
            for v in g[u]:
                if not vis[v]: vis[v] = True; stk.append(v)
        ans += prefix * t; prefix += t
    return ans

for n, e, exp in examples:
    print(dfs_iterative(n, e), "OK" if dfs_iterative(n, e) == exp else "SAI")

0 OK
14 OK


Đánh dấu `vis[v] = True` **lúc đẩy vào** ngăn xếp (không phải lúc lấy ra) để mỗi đỉnh vào ngăn xếp đúng một lần.

### 7.3 Cách 4 — Union-Find + công thức `C(n,2) − Σ C(s,2)`

In [9]:
def dsu_formula(n, edges):
    parent = list(range(n)); size = [1] * n
    def find(x):
        root = x
        while parent[root] != root: root = parent[root]
        while parent[x] != root: parent[x], x = root, parent[x]      # nén đường
        return root
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            if size[ra] < size[rb]: ra, rb = rb, ra
            parent[rb] = ra; size[ra] += size[rb]
    same = sum(size[r] * (size[r] - 1) // 2 for r in range(n) if parent[r] == r)
    return n * (n - 1) // 2 - same

for n, e, exp in examples:
    print(dsu_formula(n, e), "OK" if dsu_formula(n, e) == exp else "SAI")

0 OK
14 OK


### 7.4 Cách 5 — DSU **trực tuyến**: duy trì số cặp không tới được khi thêm từng cạnh

Ban đầu mọi đỉnh tách rời: `ans = C(n, 2)`. Gộp hai thành phần kích thước `a` và `b` biến `a·b` cặp "không tới được" thành "tới được" ⇒ `ans −= a·b`. Cho đáp án sau **mỗi** cạnh.

In [10]:
def dsu_online(n, edges, history=False):
    parent = list(range(n)); size = [1] * n; ans = n * (n - 1) // 2; hist = []
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            ans -= size[ra] * size[rb]
            if size[ra] < size[rb]: ra, rb = rb, ra
            parent[rb] = ra; size[ra] += size[rb]
        hist.append(ans)
    return hist if history else ans

for n, e, exp in examples:
    print(dsu_online(n, e), "OK" if dsu_online(n, e) == exp else "SAI", "| sau từng cạnh:", dsu_online(n, e, True))

0 OK | sau từng cạnh: [2, 0, 0]
14 OK | sau từng cạnh: [20, 18, 15, 14, 14]


### 7.5 Cách 6 — BFS + `Counter` nhãn thành phần + công thức `(n² − Σ s²) / 2`

In [11]:
def bfs_labels(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    label = [-1] * n; c = 0
    for s in range(n):
        if label[s] != -1: continue
        label[s] = c; q = deque([s])
        while q:
            u = q.popleft()
            for v in g[u]:
                if label[v] == -1: label[v] = c; q.append(v)
        c += 1
    sizes = Counter(label).values()
    return (n * n - sum(x * x for x in sizes)) // 2

for n, e, exp in examples:
    print(bfs_labels(n, e), "OK" if bfs_labels(n, e) == exp else "SAI")

0 OK
14 OK


## 8. TRACE từng bước

### 8.1 Trace DSU trực tuyến trên ví dụ 2

In [12]:
n, e, _ = examples[1]
parent = list(range(n)); size = [1] * n; ans = n * (n - 1) // 2
def find(x):
    while parent[x] != x: x = parent[x]
    return x
print("ban đầu ans = C(7,2) =", ans)
for a, b in e:
    ra, rb = find(a), find(b)
    if ra != rb:
        ans -= size[ra] * size[rb]; print(f"cạnh ({a},{b}): gộp kích thước {size[ra]} và {size[rb]} -> ans -= {size[ra] * size[rb]} = {ans}")
        if size[ra] < size[rb]: ra, rb = rb, ra
        parent[rb] = ra; size[ra] += size[rb]
    else:
        print(f"cạnh ({a},{b}): đã cùng thành phần (tạo chu trình) -> ans giữ {ans}")

ban đầu ans = C(7,2) = 21
cạnh (0,2): gộp kích thước 1 và 1 -> ans -= 1 = 20
cạnh (0,5): gộp kích thước 2 và 1 -> ans -= 2 = 18
cạnh (2,4): gộp kích thước 3 và 1 -> ans -= 3 = 15
cạnh (1,6): gộp kích thước 1 và 1 -> ans -= 1 = 14
cạnh (5,4): đã cùng thành phần (tạo chu trình) -> ans giữ 14


Cạnh `(5, 4)` nối hai đỉnh đã cùng thành phần (qua `5 − 0 − 2 − 4`) — tạo chu trình, không đổi đáp án. Đáp án cuối `14` ✔. Nhận xét: số cạnh "thừa" (tạo chu trình) không ảnh hưởng đáp án — chỉ có `n − c` cạnh gộp thật sự, với `c` là số thành phần cuối cùng.

### 8.2 Trace DFS lặp: thứ tự thăm đỉnh

In [13]:
n, e, _ = examples[1]
for comp in components(n, e): print("thành phần:", comp, "kích thước", len(comp))

thành phần: [0, 2, 4, 5] kích thước 4
thành phần: [1, 6] kích thước 2
thành phần: [3] kích thước 1


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def countPairs(self, n: int, edges: List[List[int]]) -> int:
        parent = list(range(n))
        size = [1] * n

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]        # nén đường (path halving)
                x = parent[x]
            return x

        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra                      # hợp theo kích thước
                size[ra] += size[rb]

        ans = prefix = 0
        for r in range(n):
            if parent[r] == r:                       # mỗi gốc là một thành phần
                ans += prefix * size[r]              # cặp với mọi đỉnh của các thành phần trước
                prefix += size[r]
        return ans

In [15]:
sol = Solution()
for n, e, exp in examples:
    got = sol.countPairs(n, e)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

0 | kỳ vọng 0 | OK
14 | kỳ vọng 14 | OK


Bản nộp dùng DSU vì: không cần đệ quy, không cần lưu danh sách kề (tiết kiệm bộ nhớ với `m = 2·10⁵`), và dễ mở rộng sang phiên bản trực tuyến.

## 10. Vì sao thuật toán đúng

**Bổ đề 1.** `u`, `v` không tới được nhau ⇔ `comp(u) ≠ comp(v)`. Định nghĩa của thành phần liên thông (lớp tương đương của quan hệ tới được).

**Bổ đề 2 (đếm bằng tổng chạy).** Liệt kê các thành phần `C₁, …, C_c` theo một thứ tự bất kỳ, kích thước `s₁, …, s_c`. Cặp không thứ tự `{u, v}` với `u ∈ C_i`, `v ∈ C_j`, `i < j` được đếm đúng một lần — trong số hạng `prefix · s_j` khi xử lý `C_j` (vì `u` đã nằm trong `prefix`). Cặp cùng thành phần không bao giờ được đếm. ⇒ `ans = Σ_{i<j} sᵢ sⱼ` = số cặp khác thành phần.

**Bổ đề 3 (DSU tìm đúng thành phần).** Bất biến: sau khi xử lý một tập cạnh `E'`, `find(u) = find(v)` ⇔ `u`, `v` liên thông trong đồ thị `(V, E')`, và `size[root]` = kích thước thành phần. Cạnh `(a, b)` nối hai thành phần (nếu khác nhau) thành một; gộp cây và cộng kích thước giữ đúng bất biến; nén đường không đổi gốc của bất kỳ đỉnh nào.

**DSU trực tuyến.** Gộp thành phần kích thước `a` và `b`: các cặp giữa hai thành phần (`a·b` cặp) chuyển từ "không tới được" sang "tới được"; mọi cặp khác không đổi ⇒ `ans −= a·b`. ∎

## 11. Phân tích Time / Space Complexity

`n ≤ 10⁵`, `m ≤ 2·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS mỗi cặp | O(n² · (n + m)) | O(n + m) |
| 1b. BFS mỗi đỉnh | O(n · (n + m)) | O(n + m) |
| 2. DFS đệ quy | O(n + m) | O(n + m) + ngăn xếp đệ quy O(n) |
| 3. DFS lặp | O(n + m) | O(n + m) |
| 4. DSU + công thức | O((n + m) α(n)) | O(n) |
| 5. DSU trực tuyến | O((n + m) α(n)) | O(n) |
| 6. BFS nhãn | O(n + m) | O(n + m) |

In [16]:
rng = random.Random(2316)
N = 10**5
tests = {"ngẫu nhiên m=2e5": [[rng.randrange(N), rng.randrange(N)] for _ in range(2 * 10**5)],
         "đường thẳng": [[i, i + 1] for i in range(N - 1)],
         "không cạnh": [],
         "nhiều mảnh nhỏ": [[i, i + 1] for i in range(0, N - 1, 3)]}
for label, e in tests.items():
    e = [x for x in e if x[0] != x[1]]
    res = []
    for name, f in [("dfs_iterative", dfs_iterative), ("dsu_formula", dsu_formula), ("dsu_online", dsu_online),
                    ("bfs_labels", bfs_labels), ("Solution", Solution().countPairs)]:
        t0 = time.perf_counter(); res.append(f(N, e)); dt = time.perf_counter() - t0
        print(f"{label:<17} {name:<14} {dt*1000:7.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

ngẫu nhiên m=2e5  dfs_iterative    202.3 ms -> 195744671
ngẫu nhiên m=2e5  dsu_formula      117.9 ms -> 195744671


ngẫu nhiên m=2e5  dsu_online       122.6 ms -> 195744671
ngẫu nhiên m=2e5  bfs_labels       158.0 ms -> 195744671


ngẫu nhiên m=2e5  Solution         104.7 ms -> 195744671
đường thẳng       dfs_iterative     28.1 ms -> 0
đường thẳng       dsu_formula       29.2 ms -> 0
đường thẳng       dsu_online        28.9 ms -> 0
đường thẳng       bfs_labels        73.4 ms -> 0
đường thẳng       Solution          24.5 ms -> 0
không cạnh        dfs_iterative     19.5 ms -> 4999950000


không cạnh        dsu_formula        8.7 ms -> 4999950000
không cạnh        dsu_online         1.3 ms -> 4999950000
không cạnh        bfs_labels        37.5 ms -> 4999950000
không cạnh        Solution           8.8 ms -> 4999950000
nhiều mảnh nhỏ    dfs_iterative     74.4 ms -> 4999916667
nhiều mảnh nhỏ    dsu_formula       14.1 ms -> 4999916667
nhiều mảnh nhỏ    dsu_online         8.1 ms -> 4999916667
nhiều mảnh nhỏ    bfs_labels        32.8 ms -> 4999916667
nhiều mảnh nhỏ    Solution          13.5 ms -> 4999916667


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "bfs_each": bfs_each, "dfs_recursive": dfs_recursive, "dfs_iterative": dfs_iterative,
       "dsu_formula": dsu_formula, "dsu_online": dsu_online, "bfs_labels": bfs_labels, "Solution": Solution().countPairs}
edge = examples + [
    (1, [], 0),                                     # một đỉnh: không có cặp
    (2, [], 1),
    (5, [], 10),                                    # không cạnh: C(5,2)
    (5, [[0, 1], [1, 2], [2, 3], [3, 4]], 0),       # liên thông
    (4, [[0, 1], [2, 3]], 4),                       # hai cặp: 2*2
    (6, [[0, 1], [0, 2], [3, 4]], 11),              # kích thước 3, 2, 1: 3*2 + 3*1 + 2*1
]
for n, e, exp in edge:
    for name, f in ALL.items():
        got = f(n, e)
        assert got == exp, (name, n, e, got, exp)
    print(f"n={n} edges={e!s:<36} -> {exp}")
print("edge OK")

n=3 edges=[[0, 1], [0, 2], [1, 2]]             -> 0
n=7 edges=[[0, 2], [0, 5], [2, 4], [1, 6], [5, 4]] -> 14
n=1 edges=[]                                   -> 0
n=2 edges=[]                                   -> 1
n=5 edges=[]                                   -> 10
n=5 edges=[[0, 1], [1, 2], [2, 3], [3, 4]]     -> 0
n=4 edges=[[0, 1], [2, 3]]                     -> 4
n=6 edges=[[0, 1], [0, 2], [3, 4]]             -> 11
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [18]:
rng = random.Random(42)
for _ in range(500):
    n = rng.randint(1, 12)
    allp = [[a, b] for a in range(n) for b in range(a + 1, n)]
    e = rng.sample(allp, rng.randint(0, min(len(allp), rng.choice([2, 5, 15]))))
    ref = brute(n, e)
    for name, f in ALL.items():
        assert f(n, e) == ref, (name, n, e)
print("500 đồ thị ngẫu nhiên OK")

500 đồ thị ngẫu nhiên OK


### 12.3 Tính chất: thêm cạnh không làm tăng đáp án; `0 ≤ ans ≤ C(n, 2)`; hoán vị nhãn đỉnh không đổi đáp án

In [19]:
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(1, 40); e = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 40))]
    e = [x for x in e if x[0] != x[1]]
    hist = dsu_online(n, e, True)
    assert all(a >= b for a, b in zip(hist, hist[1:]))
    ans = Solution().countPairs(n, e)
    assert 0 <= ans <= n * (n - 1) // 2
    perm = list(range(n)); rng.shuffle(perm)
    assert Solution().countPairs(n, [[perm[a], perm[b]] for a, b in e]) == ans
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: `n = 10⁵` không cạnh ⇒ `C(10⁵, 2) = 4 999 950 000` (vượt int32)

In [20]:
ans = Solution().countPairs(10**5, [])
print(ans, ans > 2**31 - 1)
assert ans == 10**5 * (10**5 - 1) // 2

4999950000 True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cặp **có thứ tự** `Σ s·(n − s)` mà quên chia 2

In [21]:
def wrong_ordered(n, edges):
    sizes = [len(c) for c in components(n, edges)]
    return sum(s * (n - s) for s in sizes)             # SAI: mỗi cặp đếm hai lần

for n, e, _ in examples:
    print("bản sai :", wrong_ordered(n, e), "| bản đúng:", Solution().countPairs(n, e))

bản sai : 0 | bản đúng: 0
bản sai : 28 | bản đúng: 14


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng (0 · 2 = 0). Ví dụ 2: `4·3 + 2·5 + 1·6 = 28 = 2 · 14`. **Sửa:** chia 2, hoặc dùng tổng chạy `prefix · t`.

### Lỗi 2 — Chỉ xét các đỉnh xuất hiện trong `edges` (quên đỉnh cô lập)

In [22]:
def wrong_isolated(n, edges):
    nodes = sorted({x for e in edges for x in e})        # SAI: bỏ đỉnh không có cạnh
    idx = {v: i for i, v in enumerate(nodes)}
    comps = components(len(nodes), [[idx[a], idx[b]] for a, b in edges])
    ans = prefix = 0
    for c in comps: ans += prefix * len(c); prefix += len(c)
    return ans

for n, e, _ in examples:
    print("bản sai :", wrong_isolated(n, e), "| bản đúng:", Solution().countPairs(n, e))

bản sai : 0 | bản đúng: 0
bản sai : 8 | bản đúng: 14


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng vì mọi đỉnh đều có cạnh. Ví dụ 2 có đỉnh 3 cô lập — một thành phần kích thước 1, không tới được 6 đỉnh còn lại. Bản sai bỏ nó ⇒ chỉ đếm `4 · 2 = 8`. **Sửa:** duyệt mọi `i ∈ [0, n)`.

### Lỗi 3 — DFS **đệ quy** trên đồ thị đường thẳng dài

In [23]:
path = [[i, i + 1] for i in range(5000)]
try:
    print("bản đệ quy:", dfs_recursive(5001, path))
except RecursionError as ex:
    print("bản đệ quy: RecursionError -", str(ex)[:50])
print("bản lặp   :", dfs_iterative(5001, path), "| bản đúng:", Solution().countPairs(5001, path))

bản đệ quy: RecursionError - maximum recursion depth exceeded
bản lặp   : 0 | bản đúng: 0


**Vì sao sai:** độ sâu đệ quy bằng độ dài đường đi DFS (tới `n − 1 = 10⁵`) vượt giới hạn mặc định 1000. `sys.setrecursionlimit` lớn có thể làm **tràn ngăn xếp C** và chết tiến trình. **Sửa:** DFS lặp, BFS, hoặc DSU.

### Lỗi 4 — Union-Find gộp **đỉnh** thay vì **gốc** (`parent[b] = a`)

In [24]:
def wrong_union(n, edges):
    parent = list(range(n))
    def find(x):
        while parent[x] != x: x = parent[x]
        return x
    for a, b in edges:
        parent[b] = a                                    # SAI: phải gộp gốc của hai cây
    sizes = Counter(find(x) for x in range(n)).values()
    return (n * n - sum(s * s for s in sizes)) // 2

for n, e in [examples[1][:2], (4, [[0, 1], [2, 1], [2, 3]])]:
    print("bản sai :", wrong_union(n, e), "| bản đúng:", Solution().countPairs(n, e))

bản sai : 14 | bản đúng: 14
bản sai : 3 | bản đúng: 0


**Vì sao sai:** ví dụ 2 **trùng hợp** đúng — mỗi đỉnh chỉ bị gán `parent` một lần làm đầu `b` của cạnh, nên không liên kết nào bị ghi đè. `(4, [[0,1],[2,1],[2,3]])`: sau `parent[1] = 0` rồi `parent[1] = 2`, đỉnh 0 bị "cắt" khỏi cây của 1 — liên kết `0 − 1` bị mất. **Sửa:** `parent[find(b)] = find(a)`.

### Lỗi 5 — Đếm số cặp **tới được** (cùng thành phần) thay vì không tới được

In [25]:
def wrong_reachable(n, edges):
    return sum(len(c) * (len(c) - 1) // 2 for c in components(n, edges))   # SAI: đây là phần bù

for n, e, _ in examples:
    print("bản sai :", wrong_reachable(n, e), "| bản đúng:", Solution().countPairs(n, e))

bản sai : 3 | bản đúng: 0
bản sai : 7 | bản đúng: 14


**Vì sao sai:** đại lượng `Σ C(s, 2)` là số cặp **cùng** thành phần — phần bù của đáp án trong tập `C(n, 2)` cặp. Ví dụ 1: tam giác có `C(3, 2) = 3` cặp tới được, đáp án là `3 − 3 = 0`; ví dụ 2: `C(4,2) + C(2,2) + 0 = 7`, đáp án `21 − 7 = 14`. **Sửa:** lấy `C(n, 2)` trừ đi đại lượng này.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cặp đỉnh (không) tới được nhau", "số nhóm", "kích thước nhóm" ⇒ **thành phần liên thông**: DFS/BFS lặp hoặc DSU.
- Đếm cặp khác nhóm: `C(n, 2) − Σ C(s, 2)` hoặc **tổng chạy** `prefix · t` — không cần O(c²) cặp nhóm.
- Python + đồ thị lớn ⇒ **tránh đệ quy**.
- Thêm cạnh dần và hỏi liên tục ⇒ **DSU trực tuyến**; xóa cạnh ⇒ xử lý **ngược** (thêm cạnh theo thứ tự ngược) bằng DSU.
- Bài cùng họ: 323 Number of Connected Components, 547 Number of Provinces, 1319 Number of Operations to Make Network Connected, 2492, 1202 Smallest String With Swaps, 2685 Count the Number of Complete Components, 1697.

## 15. Biến thể của bài

### 15.1 Biến thể A — Xóa cạnh dần, hỏi đáp án sau mỗi lần xóa (offline: đảo ngược thành thêm cạnh)

In [26]:
def after_each_removal(n, edges, removal_order):
    removed = set(map(tuple, removal_order))
    keep = [e for e in edges if tuple(e) not in removed]
    parent = list(range(n)); size = [1] * n
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    ans = n * (n - 1) // 2
    def add(a, b):
        nonlocal ans
        ra, rb = find(a), find(b)
        if ra != rb:
            ans -= size[ra] * size[rb]
            if size[ra] < size[rb]: ra, rb = rb, ra
            parent[rb] = ra; size[ra] += size[rb]
    for a, b in keep: add(a, b)
    out = [ans]                                     # trạng thái sau khi xóa hết removal_order
    for a, b in reversed(removal_order):
        add(a, b); out.append(ans)
    out.reverse()                                   # out[0] = chưa xóa gì, out[t] = sau t lần xóa
    return out

def after_each_removal_brute(n, edges, removal_order):
    cur = [list(e) for e in edges]; out = [brute(n, cur)]
    for r in removal_order:
        cur = [e for e in cur if e != list(r)]; out.append(brute(n, cur))
    return out

rng = random.Random(151)
for _ in range(200):
    n = rng.randint(1, 8); allp = [[a, b] for a in range(n) for b in range(a + 1, n)]
    e = rng.sample(allp, rng.randint(0, len(allp))); order = rng.sample(e, rng.randint(0, len(e)))
    assert after_each_removal(n, e, order) == after_each_removal_brute(n, e, order)
n, e, _ = examples[1]
print(after_each_removal(n, e, [[0, 2], [5, 4], [1, 6]]))

[14, 14, 18, 19]


### 15.2 Biến thể B — Số cặp không tới được **có trọng số đỉnh** (`Σ w_u · w_v` qua các cặp khác thành phần)

In [27]:
def weighted_pairs(n, edges, w):
    total = prefix = 0
    for comp in components(n, edges):
        t = sum(w[u] for u in comp); total += prefix * t; prefix += t
    return total

rng = random.Random(152)
for _ in range(200):
    n = rng.randint(1, 9); allp = [[a, b] for a in range(n) for b in range(a + 1, n)]
    e = rng.sample(allp, rng.randint(0, len(allp))); w = [rng.randint(0, 5) for _ in range(n)]
    comp_of = {u: i for i, c in enumerate(components(n, e)) for u in c}
    ref = sum(w[u] * w[v] for u in range(n) for v in range(u + 1, n) if comp_of[u] != comp_of[v])
    assert weighted_pairs(n, e, w) == ref
print(weighted_pairs(7, examples[1][1], [1] * 7), "(trọng số 1 => bài gốc)")

14 (trọng số 1 => bài gốc)


### 15.3 Biến thể C — Số cạnh **tối thiểu** cần thêm để mọi cặp tới được nhau = số thành phần − 1

In [28]:
def edges_to_connect(n, edges):
    return len(components(n, edges)) - 1
print(edges_to_connect(*examples[1][:2]), edges_to_connect(*examples[0][:2]))

2 0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mỗi cặp / mỗi đỉnh | định nghĩa | O(n²…) | O(n+m) | kiểm chứng |
| 2. DFS đệ quy | doocs 1 | O(n+m) | O(n+m) + đệ quy | ngôn ngữ có ngăn xếp sâu |
| 3. DFS lặp | ngăn xếp tường minh | O(n+m) | O(n+m) | an toàn trong Python |
| 4. DSU + công thức | gộp cạnh | O((n+m)α) | O(n) | **mặc định**: không cần danh sách kề |
| 5. DSU trực tuyến | `ans −= a·b` | O((n+m)α) | O(n) | thêm cạnh dần / xóa cạnh offline |
| 6. BFS nhãn + Counter | `(n² − Σs²)/2` | O(n+m) | O(n+m) | cần nhãn thành phần cho việc khác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countPairs(5, [[0, 1], [2, 3]])` và `countPairs(4, [[0, 1], [1, 2], [2, 3], [3, 0]])`?

**Bài 2 (tính tay):** `n = 8`, các thành phần có kích thước `3, 3, 1, 1`. Tính đáp án bằng cả hai công thức.

**Bài 3 (điền TODO):** Viết `bt3_todo(n, edges)` trả về **danh sách kích thước** các thành phần (giảm dần) bằng BFS.

**Bài 4 (tự viết lại):** Viết `bt4_todo(n, edges)` cài DSU + tổng chạy không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(n, edges)` trả về số cặp không tới được **sau khi thêm đúng một cạnh tối ưu** (chọn cạnh làm đáp án nhỏ nhất).

In [29]:
def bt3_todo(n, edges):
    # TODO: kích thước các thành phần
    pass

def bt4_todo(n, edges):
    # TODO: DSU + tổng chạy
    pass

def bt5_todo(n, edges):
    # TODO: thêm một cạnh tối ưu
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thành phần `{0,1}, {2,3}, {4}`. Mức 2: `C(5,2) − 1 − 1 = 8`. Mức 3: chu trình 4 đỉnh liên thông ⇒ `0`.

**Bài 2.** Mức 1: `C(8,2) = 28`. Mức 2: `Σ C(s,2) = 3 + 3 + 0 + 0 = 6` ⇒ `22`. Mức 3: tổng chạy: `0 + 3·3 + 6·1 + 7·1 = 22`.

**Bài 3.** Mức 1: danh sách kề. Mức 2: BFS từ mỗi đỉnh chưa thăm, đếm. Mức 3: `sorted(..., reverse=True)`.

**Bài 4.** Mức 1: `parent`, `size`, `find` có nén đường. Mức 2: gộp mọi cạnh. Mức 3: tổng chạy trên các gốc.

**Bài 5.** Mức 1: thêm một cạnh giữa hai thành phần `a, b` làm giảm đáp án `a·b`. Mức 2: chọn hai thành phần **lớn nhất**. Mức 3: nếu chỉ có một thành phần, đáp án không đổi (0).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `8`, `0`. **Bài 2:** `22`.

In [30]:
assert Solution().countPairs(5, [[0, 1], [2, 3]]) == 8
assert Solution().countPairs(4, [[0, 1], [1, 2], [2, 3], [3, 0]]) == 0
assert 28 - (3 + 3) == 22 == 0 + 3 * 3 + 6 * 1 + 7 * 1

def bt3(n, edges):
    g = [[] for _ in range(n)]
    for a, b in edges: g[a].append(b); g[b].append(a)
    seen = [False] * n; out = []
    for s in range(n):
        if seen[s]: continue
        seen[s] = True; q = deque([s]); cnt = 0
        while q:
            u = q.popleft(); cnt += 1
            for v in g[u]:
                if not seen[v]: seen[v] = True; q.append(v)
        out.append(cnt)
    return sorted(out, reverse=True)

def bt4(n, edges):
    return Solution().countPairs(n, edges)

def bt5(n, edges):
    sizes = bt3(n, edges); base = Solution().countPairs(n, edges)
    return base - sizes[0] * sizes[1] if len(sizes) >= 2 else base

def bt5_brute(n, edges):
    best = Solution().countPairs(n, edges)
    for a in range(n):
        for b in range(a + 1, n):
            best = min(best, Solution().countPairs(n, edges + [[a, b]]))
    return best

rng = random.Random(1996)
for _ in range(200):
    n = rng.randint(1, 9); allp = [[a, b] for a in range(n) for b in range(a + 1, n)]
    e = rng.sample(allp, rng.randint(0, min(len(allp), 6)))
    assert bt3(n, e) == sorted(map(len, components(n, e)), reverse=True)
    assert bt4(n, e) == brute(n, e)
    assert bt5(n, e) == bt5_brute(n, e)
print(bt3(*examples[1][:2]), bt5(*examples[1][:2]))
print("Đáp án bài tập: tất cả assert OK")

[4, 2, 1] 6
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
parent, size = list(range(n)), [1] * n
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in edges:
    ra, rb = find(a), find(b)
    if ra != rb:
        if size[ra] < size[rb]: ra, rb = rb, ra
        parent[rb] = ra; size[ra] += size[rb]
ans = prefix = 0
for r in range(n):
    if parent[r] == r: ans += prefix * size[r]; prefix += size[r]
return ans
```

**Checklist:**
- [ ] Đếm cặp **không thứ tự** (không nhân đôi).
- [ ] Gồm cả đỉnh cô lập (duyệt mọi `0 … n−1`).
- [ ] Không DFS đệ quy với `n = 10⁵`.
- [ ] DSU gộp **gốc**, không gộp đỉnh.